In [13]:
import pandas as pd
import numpy as np
import tensorflow as tf


from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [14]:
# 1. 데이터 불러오기
df = pd.read_csv("/content/drive/MyDrive/Colab Notebooks/abalone.csv")

print(df.head())
print(df.shape)

   id Sex  Length  Diameter  Height  Whole_weight  Shucked_weight  \
0   0   M   0.455     0.365   0.095        0.5140          0.2245   
1   1   M   0.350     0.265   0.090        0.2255          0.0995   
2   2   F   0.530     0.420   0.135        0.6770          0.2565   
3   3   M   0.440     0.365   0.125        0.5160          0.2155   
4   4   I   0.330     0.255   0.080        0.2050          0.0895   

   Viscera_weight  Shell_weight  Rings  
0          0.1010         0.150     15  
1          0.0485         0.070      7  
2          0.1415         0.210      9  
3          0.1140         0.155     10  
4          0.0395         0.055      7  
(4177, 10)


In [15]:
# 2. 입력
X = df.drop(["id", "Rings"], axis=1)
X = pd.get_dummies(X, columns=["Sex"]).values
# Sex 열은 문자열이므로 원핫 인코딩 해야함
y = df["Rings"].values
# 회귀는 연속적인 값을 예측하므로 y값 원핫 인코딩 안함

In [16]:
# 3. 학습 / 테스트 데이터 분리
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# Rings(y)가 연속형 수치 데이터이므로 클래스 비율 기반의 층화 추출을 적용할 수 없기 때문에 stratify=y 제거

In [17]:
# 4. 데이터 정규화
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [18]:
# 5. 모델 생성
model = tf.keras.Sequential()
model.add(tf.keras.layers.Dense(64, activation="relu", input_shape=(X_train.shape[1],)))
model.add(tf.keras.layers.Dense(32, activation="relu"))
model.add(tf.keras.layers.Dense(16, activation="relu"))
model.add(tf.keras.layers.Dense(1))
# 성능을 높이기 위해 은닉층 하나 더 추가
# 회귀이므로 출력층 활성화함수 제거
# X_train.shape[1]을 사용하면 입력 특성 개수가 바껴도 자동 적용
# 여기서는 Sex열이 M, F, I 세개로 원핫 인코딩 되었으므로 특성 갯수는 7+3 =10개

/usr/local/lib/python3.13/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [19]:
# 6. 모델 설정
model.compile(optimizer="adam", loss="mse", metrics=["mae"])
#회귀이므로 손실함수 mse 사용, 매트릭스 mae 사용

In [20]:
# 7. 모델 구조 출력
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                 │ (None, 64)             │           704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 32)             │         2,080 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_6 (Dense)                 │ (None, 16)             │           528 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_7 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 3,329 (13.00 KB)

 Trainable params: 3,329 (13.00 KB)

 Non-trainable params: 0 (0.00 B)

In [21]:
# 8. 모델 학습
history = model.fit(X_train, y_train, epochs=100, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 2s 5ms/step - loss: 57.9979 - mae: 6.5017 - val_loss: 19.9812 - val_mae: 3.4369
Epoch 2/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - loss: 13.5934 - mae: 2.6709 - val_loss: 6.6666 - val_mae: 1.8862
Epoch 3/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 7.0727 - mae: 1.7953 - val_loss: 4.7984 - val_mae: 1.5416
Epoch 4/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 6.0128 - mae: 1.6908 - val_loss: 4.5080 - val_mae: 1.4894
Epoch 5/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step - loss: 5.6036 - mae: 1.6534 - val_loss: 4.3632 - val_mae: 1.4660
Epoch 6/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - loss: 5.2893 - mae: 1.6224 - val_loss: 4.2709 - val_mae: 1.4920
Epoch 7/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 5.1834 - mae: 1.6096 - val_loss: 4.2858 - val_mae: 1.4634
Epoch 8/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step - loss: 5.0457 - mae: 1.5934 - val_loss: 4.1166 - val_mae: 1.4374
Epoch 9/100
84/84 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - loss: 4.

In [22]:
# 9. 테스트 데이터 예측
# 회귀 모델의 predict()는 연속된 나이테 수(Rings) 예측값을 반환하므로 .flatten()을 사용하 1차원 배열로 변환
y_pred = model.predict(X_test).flatten()

27/27 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step


In [23]:
# 10. 성능 평가
mse = mean_squared_error(y_test, y_pred)
mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("\n--- 회귀 성능 평가 결과 ---")
print(f"MAE (평균 절대 오차) : {mae:.4f}")
print(f"MSE (평균 제곱 오차) : {mse:.4f}")
print(f"R² Score (결정계수) : {r2:.4f}")


--- 회귀 성능 평가 결과 ---
MAE (평균 절대 오차) : 1.4973
MSE (평균 제곱 오차) : 4.4384
R² Score (결정계수) : 0.5900
